# Dispersion curve from `pinn_data.mat`
Load PINN response and compute an f-k style dispersion map.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

In [ ]:
mat = loadmat('../Result/pinn_data.mat')
print('keys:', [k for k in mat.keys() if not k.startswith('__')])

In [ ]:
# Robust displacement extraction from MAT file
# 1) preferred keys, 2) fallback: largest 2D numeric array
preferred = ['x', 'disp', 'u', 'response', 'X', 'U']
X = None
selected_key = None

for key in preferred:
    if key in mat:
        arr = np.array(mat[key])
        if np.issubdtype(arr.dtype, np.number) and arr.ndim >= 2:
            X = arr
            selected_key = key
            break

if X is None:
    candidates = []
    for k, v in mat.items():
        if k.startswith('__'):
            continue
        arr = np.array(v)
        if np.issubdtype(arr.dtype, np.number) and arr.ndim >= 2:
            candidates.append((k, arr))

    if not candidates:
        raise ValueError('No numeric 2D+ array found in pinn_data.mat. Please inspect keys manually.')

    # choose the candidate with largest area (likely response matrix)
    selected_key, X = max(candidates, key=lambda kv: kv[1].shape[0] * kv[1].shape[1])

print('Using displacement key:', selected_key, 'shape:', X.shape)

# Flatten higher dims if needed (keep first two axes by default)
if X.ndim > 2:
    X = np.reshape(X, (X.shape[0], -1))

# enforce [time, space] orientation heuristically
if X.shape[0] < X.shape[1]:
    X = X.T

X = X - X.mean(axis=0, keepdims=True)
nt, nx = X.shape
print('response shape [time, space]:', X.shape)

In [ ]:
# Optional defaults if dt/dx are not stored in MAT
dt = float(np.squeeze(mat['dt'])) if 'dt' in mat else 1.0
dx = float(np.squeeze(mat['dx'])) if 'dx' in mat else 1.0

S = np.fft.fftshift(np.fft.fft2(X))
P = np.abs(S)**2

f = np.fft.fftshift(np.fft.fftfreq(nt, d=dt))
k = 2*np.pi*np.fft.fftshift(np.fft.fftfreq(nx, d=dx))

In [ ]:
plt.figure(figsize=(8,5))
plt.pcolormesh(k, f, 10*np.log10(P / (P.max()+1e-12) + 1e-12), shading='auto', cmap='magma')
plt.xlabel('Wavenumber k [rad/m or index-based]')
plt.ylabel('Frequency f [Hz or index-based]')
plt.title('Dispersion-style f-k energy map from PINN response')
plt.colorbar(label='Power (dB, normalized)')
plt.tight_layout()
plt.show()